## Step 1: Environment Setup & Spark Session

In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("SmartCityBusClustering_Cleaning") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark version:", spark.version)
print("Driver Python:", sys.executable)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/24 08:34:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/24 08:34:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1
Driver Python: /opt/anaconda3/bin/python


## Step 2: Project Paths

In [2]:
PROJECT_ROOT = "/Users/aayushbohara/Desktop/smartcity-bus-clustering"
RAW_COMBINED_PATH = os.path.join(PROJECT_ROOT, "data/processed/combined_raw_dataset.csv")
CLEANED_OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data/processed/cleaned_dataset.csv")

print("Raw combined path:", RAW_COMBINED_PATH, "| exists:", os.path.exists(RAW_COMBINED_PATH))

Raw combined path: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/combined_raw_dataset.csv | exists: True


## Step 3: Load Raw Combined Dataset with an Explicit Schema

In [3]:
raw_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("lineRef", StringType(), True),
    StructField("directionRef", StringType(), True),
    StructField("vehicleRef", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("location_source_file", StringType(), True),   # renamed from source_file
    StructField("lineName", StringType(), True),
    StructField("serviceCode", StringType(), True),
    StructField("operator", StringType(), True),
    StructField("nationalOperatorCode", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("destination", StringType(), True),
    StructField("vehicleJourneyCode", StringType(), True),
    StructField("departureTime", TimestampType(), True),
    StructField("journeyPatternRef", StringType(), True),
    StructField("timetable_source_file", StringType(), True),  # renamed from source_file
    StructField("fare_organisation", StringType(), True),
    StructField("common_product_type", StringType(), True),
    StructField("common_tariff_basis", StringType(), True),
    StructField("common_product_name", StringType(), True),
    StructField("fare_product_count", IntegerType(), True),
    StructField("disruption_count", IntegerType(), True),
])

raw_df = spark.read.csv(RAW_COMBINED_PATH, header=True, schema=raw_schema)

print("Raw combined row count:", raw_df.count())
raw_df.printSchema()
raw_df.show(5, truncate=False)

Raw combined row count: 771733
root
 |-- timestamp: timestamp (nullable = true)
 |-- lineRef: string (nullable = true)
 |-- directionRef: string (nullable = true)
 |-- vehicleRef: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location_source_file: string (nullable = true)
 |-- lineName: string (nullable = true)
 |-- serviceCode: string (nullable = true)
 |-- operator: string (nullable = true)
 |-- nationalOperatorCode: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- vehicleJourneyCode: string (nullable = true)
 |-- departureTime: timestamp (nullable = true)
 |-- journeyPatternRef: string (nullable = true)
 |-- timetable_source_file: string (nullable = true)
 |-- fare_organisation: string (nullable = true)
 |-- common_product_type: string (nullable = true)
 |-- common_tariff_basis: string (nullable = true)
 |-- common_product_name: string (nullable = true)
 |--

26/07/24 08:35:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
 Schema: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, location_source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, timetable_source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
Expected: location_source_file but found: source_file
CSV file: file:///Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/combined_raw_dataset.csv/part-00003-b122674e-82a0-4446-b

## Step 4: Data Quality Audit

In [4]:
from pyspark.sql.functions import sum as spark_sum, when, count, min as spark_min, max as spark_max

# 1. Null counts per column
null_counts = raw_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in raw_df.columns
])
print("=== NULL COUNTS PER COLUMN ===")
null_counts.show(truncate=False, vertical=True)

# 2. Duplicate row check (exact full-row duplicates)
total_rows = raw_df.count()
distinct_rows = raw_df.distinct().count()
print(f"\nTotal rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Exact duplicate rows: {total_rows - distinct_rows}")

# 3. GPS coordinate range check (Greater Manchester bounding box)
print("\n=== LATITUDE / LONGITUDE RANGE ===")
raw_df.select(
    spark_min("latitude").alias("min_lat"), spark_max("latitude").alias("max_lat"),
    spark_min("longitude").alias("min_lon"), spark_max("longitude").alias("max_lon")
).show()

out_of_range = raw_df.filter(
    (col("latitude") < 53.0) | (col("latitude") > 54.0) |
    (col("longitude") < -3.0) | (col("longitude") > -1.5)
).count()
print(f"Rows with GPS coordinates outside expected Greater Manchester range: {out_of_range}")

# 4. Timestamp sanity check
print("\n=== TIMESTAMP RANGE ===")
raw_df.select(
    spark_min("timestamp").alias("earliest"), spark_max("timestamp").alias("latest")
).show(truncate=False)

=== NULL COUNTS PER COLUMN ===


26/07/24 08:36:44 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
 Schema: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, location_source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, timetable_source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
Expected: location_source_file but found: source_file
CSV file: file:///Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/combined_raw_dataset.csv/part-00002-b122674e-82a0-4446-b

-RECORD 0-----------------------
 timestamp             | 0      
 lineRef               | 0      
 directionRef          | 0      
 vehicleRef            | 0      
 latitude              | 0      
 longitude             | 0      
 location_source_file  | 0      
 lineName              | 2423   
 serviceCode           | 2423   
 operator              | 2423   
 nationalOperatorCode  | 2423   
 origin                | 2423   
 destination           | 2423   
 vehicleJourneyCode    | 2423   
 departureTime         | 2423   
 journeyPatternRef     | 2423   
 timetable_source_file | 2423   
 fare_organisation     | 244893 
 common_product_type   | 244893 
 common_tariff_basis   | 244893 
 common_product_name   | 244893 
 fare_product_count    | 244893 
 disruption_count      | 0      



26/07/24 08:36:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
 Schema: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, location_source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, timetable_source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
Expected: location_source_file but found: source_file
CSV file: file:///Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/combined_raw_dataset.csv/part-00002-b122674e-82a0-4446-b


Total rows: 771733
Distinct rows: 771733
Exact duplicate rows: 0

=== LATITUDE / LONGITUDE RANGE ===
+---------+---------+---------+---------+
|  min_lat|  max_lat|  min_lon|  max_lon|
+---------+---------+---------+---------+
|53.338156|53.660219|-2.723324|-1.782319|
+---------+---------+---------+---------+

Rows with GPS coordinates outside expected Greater Manchester range: 0

=== TIMESTAMP RANGE ===
+-------------------+-------------------+
|earliest           |latest             |
+-------------------+-------------------+
|2026-07-21 20:51:47|2026-07-23 06:21:13|
+-------------------+-------------------+



## Step 5: Cleaning Transformations

In [5]:
from pyspark.sql.functions import trim, upper, coalesce, lit

cleaned_df = raw_df \
    .withColumn("lineRef", trim(col("lineRef"))) \
    .withColumn("nationalOperatorCode", upper(trim(col("nationalOperatorCode")))) \
    .withColumn("fare_product_count", coalesce(col("fare_product_count"), lit(0))) \
    .withColumn("disruption_count", coalesce(col("disruption_count"), lit(0)))

# flag rows with no timetable match / no fares match, instead of dropping them
cleaned_df = cleaned_df.withColumn("has_timetable_match", col("lineName").isNotNull())
cleaned_df = cleaned_df.withColumn("has_fares_match", col("fare_organisation").isNotNull())

print("Cleaned row count:", cleaned_df.count())
cleaned_df.select("has_timetable_match", "has_fares_match").groupBy("has_timetable_match", "has_fares_match").count().show()

Cleaned row count: 771733
+-------------------+---------------+------+
|has_timetable_match|has_fares_match| count|
+-------------------+---------------+------+
|               true|          false|242470|
|               true|           true|526840|
|              false|          false|  2423|
+-------------------+---------------+------+



## Step 6: Export Cleaned Dataset

In [6]:
cleaned_df.coalesce(4).write.mode("overwrite").option("header", "true").csv(CLEANED_OUTPUT_PATH)
print("Exported to:", CLEANED_OUTPUT_PATH)

import glob
written_files = glob.glob(os.path.join(CLEANED_OUTPUT_PATH, "*.csv"))
print(f"Part-files written: {len(written_files)}")
for f in written_files:
    print(" -", os.path.basename(f), f"({os.path.getsize(f)/1024/1024:.1f} MB)")

26/07/24 08:39:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
 Schema: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, location_source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, timetable_source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
Expected: location_source_file but found: source_file
CSV file: file:///Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/combined_raw_dataset.csv/part-00003-b122674e-82a0-4446-b

Exported to: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/cleaned_dataset.csv
Part-files written: 4
 - part-00000-c86e3159-2053-4072-b2c9-4a30536e462f-c000.csv (55.5 MB)
 - part-00001-c86e3159-2053-4072-b2c9-4a30536e462f-c000.csv (83.5 MB)
 - part-00002-c86e3159-2053-4072-b2c9-4a30536e462f-c000.csv (55.5 MB)
 - part-00003-c86e3159-2053-4072-b2c9-4a30536e462f-c000.csv (65.9 MB)


## Step 7: Final Verification (Read-Back Check)

In [7]:
verify_df = spark.read.csv(CLEANED_OUTPUT_PATH, header=True, inferSchema=True)
print("Read-back row count:", verify_df.count())
verify_df.printSchema()

Read-back row count: 771733
root
 |-- timestamp: timestamp (nullable = true)
 |-- lineRef: string (nullable = true)
 |-- directionRef: string (nullable = true)
 |-- vehicleRef: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location_source_file: string (nullable = true)
 |-- lineName: string (nullable = true)
 |-- serviceCode: string (nullable = true)
 |-- operator: string (nullable = true)
 |-- nationalOperatorCode: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- vehicleJourneyCode: string (nullable = true)
 |-- departureTime: timestamp (nullable = true)
 |-- journeyPatternRef: string (nullable = true)
 |-- timetable_source_file: string (nullable = true)
 |-- fare_organisation: string (nullable = true)
 |-- common_product_type: string (nullable = true)
 |-- common_tariff_basis: string (nullable = true)
 |-- common_product_name: string (nullable = true)
 |-- fa

## Step 9: Single-File Convenience Export

In [9]:
SINGLE_FILE_PATH = os.path.join(PROJECT_ROOT, "data/processed/cleaned_dataset_single.csv")

cleaned_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(SINGLE_FILE_PATH)

import glob
single_file = glob.glob(os.path.join(SINGLE_FILE_PATH, "*.csv"))[0]
final_single_path = os.path.join(PROJECT_ROOT, "data/processed/cleaned_dataset_final.csv")
os.rename(single_file, final_single_path)
print("Single clean CSV ready at:", final_single_path)

26/07/24 09:00:31 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
 Schema: timestamp, lineRef, directionRef, vehicleRef, latitude, longitude, location_source_file, lineName, serviceCode, operator, nationalOperatorCode, origin, destination, vehicleJourneyCode, departureTime, journeyPatternRef, timetable_source_file, fare_organisation, common_product_type, common_tariff_basis, common_product_name, fare_product_count, disruption_count
Expected: location_source_file but found: source_file
CSV file: file:///Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/combined_raw_dataset.csv/part-00003-b122674e-82a0-4446-b

Single clean CSV ready at: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/cleaned_dataset_final.csv
